<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

Code based on https://github.com/pytorch/examples/blob/master/mnist/main.py

In this exercise, we are using high-level abstractions from torch.nn like nn.Linear.
Note: during the next lab session we will go one level deeper and implement more things
with bare hands.

Tasks:

 1. Read the code.

 2. Check that the given implementation reaches 95% test accuracy for architecture input-128-128-10 after few epochs.

 3. Add the option to use SGD with momentum instead of ADAM.

 4. Experiment with different learning rates. Use the provided TrainingVisualizer
 to plot the learning curves and gradient-to-weight ratios. Compare visualizations
 for different learning rates for both ADAM and SGD with momentum.

 5. Parameterize the constructor by a list of sizes of hidden layers of the MLP.
 Note that this requires creating a list of layers as an attribute of the Net class,
 and one can't use a standard Python list containing nn.Modules (why?).
 Check torch.nn.ModuleList.

If you run this notebook locally then you may need to install some packages.
It may be achieved by adding the following code cell to the notebook and running it:
```
!pip install torch torchvision plotly ipywidgets
```

This notebook will work faster on a Colab GPU (or TPU).
However, start with tests just on CPU, and try to end your GPU sessions as soon as you stop needing them (otherwise you may use all your free GPU time for this week).

In [1]:
import math
import sys

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import plotly.graph_objects as go
from torch import Tensor
from torchvision import datasets, transforms

# Allow custom (non-ipywidget) widgets.
if "google.colab" in sys.modules:
    from google.colab import output

    output.enable_custom_widget_manager()

# For reproducibility.
torch.manual_seed(1)

## Tensors
In ML, *tensor* is just a short name for an n-dimensional array.

In other areas like physics, it might mean a multilinear operation that can be represented, after a choice of basis, by an ndarray; this is where we would see all the extra stuff (like covariance/contravariance), like for example on the Wikipedia article for "tensor". For us, it's usually just a bunch of numbers.

torch.Tensor is can be thought of as a wrapper around numpy ndarray, but:
* the interface is slightly different (like `np.expand_dims(x, dim=n-1)` vs `torch.unsqueeze(x, axis=-1)`);
* the data can be stored in GPU memory (or others);
* it can be part of a computation graph: a computed Tensor has pointers to other Tensors it was computed from, and to functions computing partial derivatives.

All the operations defined for a torch.Tensor (`@`, `.arctan`, `.unsqueeze`, and a hundred others) can also run on a GPU and other backends.

In [2]:
x_np = np.array([[0, 1, 2], [3, 4, 5]])
x = torch.tensor(
    [[0, 1, 2], [3, 4, 5]]
)  # Also: torch.rand, ones, zeros, rand_like, ...
display(x_np, x)

array([[0, 1, 2],
       [3, 4, 5]])

tensor([[0, 1, 2],
        [3, 4, 5]])

In [3]:
torch.from_numpy(x_np).equal(x), np.array_equal(x_np, x.numpy())

(True, True)

In [4]:
x.shape, x.dtype, x.device, x.requires_grad

(torch.Size([2, 3]), torch.int64, device(type='cpu'), False)

Most operations have an in-place variant with the same name, but with an underscore.
This forgets the original value, making the computation of gradients impossible, so most of the time we don't use in-place operations.

In [5]:
x.add_(2)
display(x)

tensor([[2, 3, 4],
        [5, 6, 7]])

## Autograd
Operations on torch tensors remember the computation graph.
This allows to compute the gradient of e.g. a loss node over all other nodes with loss.backward().

In [6]:
x = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], requires_grad=True)
y = torch.sigmoid(x)
loss = ((y - 1) ** 2).mean()
print("x", x.grad_fn, ", y", y.grad_fn, ", loss", loss.grad_fn)
print(loss.grad_fn.next_functions)  # type: ignore
print(loss.grad_fn.next_functions[0][0].next_functions)  # type: ignore

x None , y <SigmoidBackward0 object at 0x000001C2376CB6A0> , loss <MeanBackward0 object at 0x000001C2375A8D00>
((<PowBackward0 object at 0x000001C2376600D0>, 0),)
((<SubBackward0 object at 0x000001C2071028C0>, 0),)


The method Tensor.backward():
* for each node with `requires_grad`: computes the gradients from each .grad_fn and propagates it using the chain rule.
* for each leaf node (`requires_grad` with no predecessors that `requires_grad`): accumulates the gradient into the tensor’s `.grad` attribute.



In [7]:
loss.backward()
x.grad

tensor([[-4.1667e-02, -1.7626e-02, -4.1719e-03],
        [-7.1418e-04, -1.0590e-04, -1.4831e-05]])

We say *accumulates* not *stores*, because if .grad already exists, `backward()` adds to it, instead of replacing it.

In [8]:
loss2 = (x * 0).mean()
loss2.backward()
x.grad

tensor([[-4.1667e-02, -1.7626e-02, -4.1719e-03],
        [-7.1418e-04, -1.0590e-04, -1.4831e-05]])

During (for example) evaluation, you don't need to store the computational graph, so you can disable it:

In [9]:
with torch.no_grad():
    y = x * x

print(y.grad_fn)

None


In some situations you may need more fine-grained control.
You can detach a tensor from the graph, to treat it like a constant tensor.

In [10]:
y = x * x
print(y.grad_fn, y.requires_grad)
y = y.detach()  # Not in-place, returns a new tensor with the same data.
print(y.grad_fn, y.requires_grad)

<MulBackward0 object at 0x000001C2378C88B0> True
None False


## Modules
A torch **module** (not *model*) is essentially just an object with a `forward()` method, usually `forward(x: Tensor) -> Tensor`.
A network (model) is built as a large module that contains and calls smaller modules, which themselves contain even smaller modules.

Basically any callable could be a module, but subclassing `nn.Module` adds features like:
* automatically registering and enumerating submodules and trainable parameters,
* serializing and storing parameters into files,
* adding hooks in the middle of existing models.

In [11]:
class MyLinear(nn.Module):
    """
    This is essentially the same as nn.Linear.

    Compare: https://github.com/pytorch/pytorch/blob/v2.9.0/torch/nn/modules/linear.py#L53
    """

    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty((out_features, in_features)))
        self.bias = nn.Parameter(torch.empty(out_features))

        self.reset_parameters()

    def reset_parameters(self) -> None:
        # The initialization used by PyTorch by default, including non-zero biases.
        bound = 1 / math.sqrt(self.in_features)
        nn.init.uniform_(self.weight, -bound, bound)
        nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, input: Tensor) -> Tensor:
        """
        Input shape: (B*, in_features).
        Output shape: (B*, out_features).
        """
        return input @ self.weight.T + self.bias


We define the module behaviour in the **`forward()`** method, not in the special `__call__()`.<br>
However, we then use the module directly as a callable, `module()`.<br>
This will call `module.forward()`, but can also calls hooks, if any are defined.

In [12]:
module = MyLinear(2, 3)
x = torch.rand((7, 5, 2))
y = module(x)
y.shape

torch.Size([7, 5, 3])

The class **nn.Parameter** just wraps a tensor to indicate that this is a trainable parameter, and changes the default `requires_grad`.<br>
When an nn.Parameter is assigned to a field of an nn.Module, it automatically registers it,
so that all parameters of this modules and its submodules can be easily accessed.

In [13]:
for i, (name, param) in enumerate(module.named_parameters()):
    print(f'{i}. "{name}":\t{param}\n')

0. "weight":	Parameter containing:
tensor([[ 0.3643, -0.3121],
        [-0.1371,  0.3319],
        [-0.6657,  0.4241]], requires_grad=True)

1. "bias":	Parameter containing:
tensor([-0.1455,  0.3597,  0.0983], requires_grad=True)



## A basic Net

In [14]:
class Net(nn.Module):
    def __init__(self, hidden_sizes: list[int] = (128, 128)):
        """
        hidden_sizes: sizes of the hidden layers, e.g. [128, 128] gives 784-128-128-10.
        """
        super().__init__()
        sizes = [784] + list(hidden_sizes) + [10]  # e.g. [784, 128, 128, 10]

        # nn.ModuleList (not a plain Python list), so the layers are registered as submodules.
        self.layers = nn.ModuleList()
        for i in range(len(sizes) - 1):
            self.layers.append(nn.Linear(sizes[i], sizes[i + 1]))

    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (B, 28, 28).
        Output: class log-probabilities (after log_softmax), shape (B, 10).
        """
        x = torch.flatten(x, start_dim=1)  # (B, 784)
        for layer in self.layers[:-1]:  # hidden layers: Linear + ReLU
            x = F.relu(layer(x))  # (B, hidden_size)
        x = self.layers[-1](x)  # output layer, no ReLU: (B, 10)
        output = F.log_softmax(x, dim=1)
        return output

**Why `nn.ModuleList` and not a plain Python list?**

`nn.Module` only registers submodules that are assigned directly as attributes, or that live inside an `nn.ModuleList` / `nn.Sequential`. Layers stored in a plain `list` are invisible to PyTorch:
- `model.parameters()` does not return them, so **the optimizer never trains them**;
- `model.to(device)` does not move them to the GPU;
- `model.state_dict()` does not save them.

Quick check:

In [15]:
class NetWithPlainList(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(784, 128), nn.Linear(128, 10)]  # plain list


class NetWithModuleList(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(784, 128), nn.Linear(128, 10)])


print("Parameters with plain list:", len(list(NetWithPlainList().parameters())))
print("Parameters with ModuleList:", len(list(NetWithModuleList().parameters())))

Parameters with plain list: 0
Parameters with ModuleList: 4


Assigning to a nn.Module to a field of nn.Module automatically registers it as a submodule.<br>
You can iterate over all submodules (recursively) with `.named_modules()` or over direct children only with `.named_children()`.

In [16]:
net = Net()
for i, (name, submodule) in enumerate(net.named_modules()):
    print(f'{i}. "{name}":\t{submodule}')

0. "":	Net(
  (layers): ModuleList(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): Linear(in_features=128, out_features=128, bias=True)
    (2): Linear(in_features=128, out_features=10, bias=True)
  )
)
1. "layers":	ModuleList(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): Linear(in_features=128, out_features=128, bias=True)
  (2): Linear(in_features=128, out_features=10, bias=True)
)
2. "layers.0":	Linear(in_features=784, out_features=128, bias=True)
3. "layers.1":	Linear(in_features=128, out_features=128, bias=True)
4. "layers.2":	Linear(in_features=128, out_features=10, bias=True)


The `.train()` and `.eval()` methods set the `training` flag of a module and all its submodules.<br>
This changes the behaviour of certain modules like BatchNorm and Dropout (see next labs).

Forgetting to switch the training mode is a common bug that can cause weird behaviour.

In [17]:
net.train()
print(net.layers[0].training)
net.eval()
print(net.layers[0].training)

True
False


Note that operations on a module (`.train()`, `.eval()`, `.to(dtype)`, `.to(device)`) modify it in-place.<br>
Operations on a tensor without a trailing underscore (including `.to(dtype)`, `.to(device)`) return a new tensor and leave the original unchanged


## Train and test code

In [18]:
# @title <small>Helper class: TrainingVisualizer</small>


class TrainingVisualizer:
    def __init__(self, log_interval: int = 10):
        self.log_interval = log_interval
        self.train_loss_fig = self.init_line_plot(
            title="Training loss", xaxis_title="Step"
        )
        self.grad_to_weight_fig = self.init_line_plot(
            title="Gradient standard deviation to weight standard deviation ratio at 1st layer",
            xaxis_title="Step",
            yaxis_title="Gradient to weight ratio (log scale)",
            yaxis_type="log",
        )
        self.test_acc_fig = self.init_line_plot(
            title="Test accuracy", x=[], xaxis_title="Epoch", mode="lines+markers"
        )

        # Parameters related to current tracked model and its training
        self.first_linear_layer = None
        self.lr = None
        self.trace_idx = -1

    def init_line_plot(
        self,
        title: str,
        x=None,
        xaxis_title: str | None = None,
        yaxis_title: str | None = None,
        yaxis_type: str = "linear",
        mode: str = "lines",
    ) -> go.FigureWidget:
        fig = go.Figure()
        fig.update_layout(
            title=title,
            title_x=0.5,
            xaxis_title=xaxis_title,
            yaxis_title=yaxis_title,
            height=400,
            width=1500,
            margin=dict(b=10, t=60),
        )
        fig.update_yaxes(type=yaxis_type)
        # We cannot add new traces dynamically because Colab has a problem with widgets
        # from plotly (traces added dynamically are rendered twice).
        # As an ugly workaround we create a lot of empty traces and update them later
        # with actual data. Empty traces are not plotted.
        for _ in range(25):
            fig.add_trace(go.Scatter(x=x, y=[], showlegend=True, mode=mode))

        fig_widget = go.FigureWidget(fig)
        display(fig_widget)
        return fig_widget

    def track_model(
        self, model: torch.nn.Module, optimizer: torch.optim.Optimizer, lr: float
    ) -> None:
        """
        Start tracking training metrics for a new model.
        """

        for field in model.__dict__["_modules"].values():
            if isinstance(field, nn.Linear):
                self.first_linear_layer = field
                break
            elif isinstance(field, nn.ModuleList):
                self.first_linear_layer = field[0]
                break

        self.lr = lr
        self.trace_idx += 1

        optim_name = type(optimizer).__name__
        self.train_loss_fig.data[self.trace_idx].name = f"{optim_name}, {lr}"
        self.grad_to_weight_fig.data[self.trace_idx].name = f"{optim_name}, {lr}"
        self.test_acc_fig.data[self.trace_idx].name = f"{optim_name}, {lr}"

    def plot_gradients_and_loss(self, batch_idx: int, loss: float) -> None:
        if batch_idx % self.log_interval == 0:
            self.train_loss_fig.data[self.trace_idx].y += (loss,)

            layer = self.first_linear_layer
            grad_to_weight_ratio = (
                self.lr * layer.weight.grad.std() / layer.weight.std()
            ).item()

            self.grad_to_weight_fig.data[self.trace_idx].y += (grad_to_weight_ratio,)

    def plot_accuracy(self, epoch: int, accuracy: float) -> None:
        self.test_acc_fig.data[self.trace_idx].x += (epoch,)
        self.test_acc_fig.data[self.trace_idx].y += (accuracy,)

In [19]:
def train_epoch(
    model: torch.nn.Module,
    device: torch.device,
    train_loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    log_interval: int,
    visualizer: TrainingVisualizer,
    verbose: bool = False,
) -> None:
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        # Move data to the device (e.g. GPU #0) where model parameters are.
        data, target = data.to(device), target.to(device)

        # Since `backward()` will accumuate gradients, remember to reset them.
        optimizer.zero_grad()

        # Forward pass.
        output = model(data)  # Logs of class probabilites, shape (B, 10)
        loss = F.nll_loss(output, target)  # Shape (1,)

        # Backward pass.
        loss.backward()

        visualizer.plot_gradients_and_loss(batch_idx, loss.item())

        # For each parameter, updates the tensor's value using its .grad.
        optimizer.step()

        if batch_idx % log_interval == 0:
            if verbose:
                done, total = batch_idx * len(data), len(train_loader.dataset)
                print(
                    f"Train Epoch: {epoch} [{done}/{total} images ({done / total:.0%})]\t"
                    + f"Loss: {loss.item():.6f}"
                )

In [20]:
def test(
    model: torch.nn.Module,
    device: torch.device,
    test_loader: torch.utils.data.DataLoader,
    epoch: int,
    visualizer: TrainingVisualizer,
    verbose: bool = False,
) -> None:
    model.eval()
    test_loss = 0
    n_correct = 0
    n_total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            # Accumulate sum of dataitem losses and only divide by n_total at the end.
            test_loss += F.nll_loss(output, target, reduction="sum").item()
            pred = output.argmax(dim=1)
            n_correct += (pred == target).sum().item()
            n_total += len(target)

    test_loss /= n_total

    if verbose:
        print(
            f"\nTest loss: {test_loss:.4f}, Accuracy: {n_correct}/{n_total} ({n_correct / n_total:.0%})\n"
        )
    visualizer.plot_accuracy(epoch, 100.0 * n_correct / n_total)
    return 100.0 * n_correct / n_total


## Hyperparameters

In [21]:
# Training uses more memory than test (due to gradient computations),
# so we can set test_batch_size to be larger.
epochs = 5
batch_size = 256
test_batch_size = 1000
lr = 1e-2
optimizer_name = "adam"  # "adam" or "sgd" (SGD with momentum)
momentum = 0.9  # only used by SGD
hidden_sizes = [128, 128]  # architecture input-128-128-10
log_interval = 10

# Check for CUDA / TPU (Google) / MPS (Apple) / XPU (Intel) / ... accelerator.
# (TPU as of 2026 gives spurious warnings about lss.item() accessing .grad - ignore them).
device = (
    torch.accelerator.current_accelerator(check_available=True)  #
    or torch.device("cpu")
)
use_accel = device != torch.device("cpu")
print(use_accel, device)

False cpu


## Dataset and dataloader
Any sequence of dataitems (and even some iterables) can be used as a ***dataset***.<br>
A dataitem can be for example an (input image, output label) pair.<br>
Torch datasets usually allow passing a transformation: it is then applied whenever you retrieve an item with `dataset[i]`.

A ***dataloader*** manages parallel loading (it can create worker threads that call `dataset[i]` in parallel), batching, shuffling, sometimes moving data between devices.

In [22]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(mean=(0.1307,), std=(0.3081,))]
)
train_dataset = datasets.MNIST(
    "../data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST("../data", train=False, transform=transform)
len(train_dataset), len(test_dataset)

(60000, 10000)

In [23]:
kwargs = {"num_workers": 1, "pin_memory": True} if use_accel else {}
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, **kwargs
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=test_batch_size, **kwargs
)

## Running it all

In [24]:
def make_optimizer(
    name: str, params, lr: float, momentum: float = 0.9
) -> torch.optim.Optimizer:
    if name == "adam":
        return optim.Adam(params, lr=lr)
    elif name == "sgd":
        return optim.SGD(params, lr=lr, momentum=momentum)
    else:
        raise ValueError(f"Unknown optimizer: {name}")

In [25]:
model = Net(hidden_sizes).to(device)

optimizer = make_optimizer(optimizer_name, model.parameters(), lr, momentum)

visualizer = TrainingVisualizer(log_interval=log_interval)
visualizer.track_model(model, optimizer, lr)

for epoch in range(1, epochs + 1):
    train_epoch(
        model,
        device,
        train_loader,
        optimizer,
        epoch,
        log_interval,
        visualizer,
        verbose=True,
    )
    test(model, device, test_loader, epoch, visualizer, verbose=True)

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'bd8f223e-b62d-4a3d-ae33-4573e9a62da3',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'e8b291ed-1e9c-4a56-bdd9-55d49bac628c',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '8e8ab0b1-22d0-4a6b-b285-ad24da9d4632',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '6bf38867-0deb-4313-8066-fb3d3dfecaaf',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '37b34264-d3c4-4a17-ace4-a41cb8aab737',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '781d5e52-9566-4969-8eb1-3954c4152348',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2b04b72f-0c3e-49c6-aa66-c69c2f3da435',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '7cee298f-6c63-464c-8026-ec438c4a238e',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '5e06135c-c620-4fc4-9ff7-babfccbf3749',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'd48a4dbd-ba14-4a0a-8cff-4ecb234b26f2',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '147b0248-f053-47d5-b3db-b9a9686631b4',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b64a7464-f997-48d8-b453-326c4a469aef',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '875736c1-b3da-4c6d-93c4-483d00629559',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2323d206-ae76-46d9-bf0b-dfbcf63acd00',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '8

Train Epoch: 1 [0/60000 images (0%)]	Loss: 2.310226
Train Epoch: 1 [2560/60000 images (4%)]	Loss: 0.559434
Train Epoch: 1 [5120/60000 images (9%)]	Loss: 0.318472
Train Epoch: 1 [7680/60000 images (13%)]	Loss: 0.306956
Train Epoch: 1 [10240/60000 images (17%)]	Loss: 0.285264
Train Epoch: 1 [12800/60000 images (21%)]	Loss: 0.181058
Train Epoch: 1 [15360/60000 images (26%)]	Loss: 0.286839
Train Epoch: 1 [17920/60000 images (30%)]	Loss: 0.245047
Train Epoch: 1 [20480/60000 images (34%)]	Loss: 0.217379
Train Epoch: 1 [23040/60000 images (38%)]	Loss: 0.209807
Train Epoch: 1 [25600/60000 images (43%)]	Loss: 0.226209
Train Epoch: 1 [28160/60000 images (47%)]	Loss: 0.291276
Train Epoch: 1 [30720/60000 images (51%)]	Loss: 0.203727
Train Epoch: 1 [33280/60000 images (55%)]	Loss: 0.165834
Train Epoch: 1 [35840/60000 images (60%)]	Loss: 0.258244
Train Epoch: 1 [38400/60000 images (64%)]	Loss: 0.151854
Train Epoch: 1 [40960/60000 images (68%)]	Loss: 0.186087
Train Epoch: 1 [43520/60000 images (73%)]

## Experiments: learning rates (Adam vs. SGD with momentum)
A single `TrainingVisualizer` is used, so every run appears as a separate line on the same plots.

The gradient-to-weight ratio is `lr * std(grad) / std(weights)` at the first layer: roughly how much the weights change per step, relative to their size. A common rule of thumb is that around **1e-3** is healthy.

In [26]:
def run_experiment(optimizer_name: str, lr: float, visualizer: TrainingVisualizer) -> list[float]:
    """Trains a fresh Net and returns the test accuracy after each epoch."""
    model = Net(hidden_sizes).to(device)
    optimizer = make_optimizer(optimizer_name, model.parameters(), lr, momentum)
    visualizer.track_model(model, optimizer, lr)
    accuracies = []
    for epoch in range(1, epochs + 1):
        train_epoch(model, device, train_loader, optimizer, epoch, log_interval, visualizer)
        accuracies.append(test(model, device, test_loader, epoch, visualizer))
    print(f"{optimizer_name:>4}, lr={lr:<6g} -> test accuracy per epoch: {[round(a, 2) for a in accuracies]}")
    return accuracies

In [27]:
torch.manual_seed(1)
lr_visualizer = TrainingVisualizer(log_interval=log_interval)

experiments = [
    ("adam", 1e-1), ("adam", 1e-2), ("adam", 1e-3), ("adam", 1e-4),
    ("sgd", 1.0), ("sgd", 1e-1), ("sgd", 1e-2), ("sgd", 1e-3),
]
results = {}
for opt_name, exp_lr in experiments:
    results[(opt_name, exp_lr)] = run_experiment(opt_name, exp_lr, lr_visualizer)

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '8b79ec71-5ed3-4cef-9f7c-db9291c98743',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'e2b4045c-3cbd-4851-9f5b-2e552e120d26',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'd9b1b5dc-619c-4197-914d-547e345addd5',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'eaba8f51-ca99-4b1e-948b-772d9dffc4d0',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '4d7f599a-a158-4d86-ba55-7b441087670e',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '00fb4a8e-306c-4534-982b-92790cb6e8d3',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2a08534d-1740-42ed-b8ca-a7f0f6eec9dc',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': '3c2c64ca-0123-4fde-82c7-d80c6e8132b4',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b89293f4-2087-4e11-bfdf-f7653de1467e',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c181ac53-b737-4821-8e77-5665220b0439',
              'y': []},
             {'mode': 'lines',
              'showlegend': True,
     

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '4ef5a4d4-cd18-4590-b79d-2d92ac21162a',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '19643c07-51be-434a-80e5-a8850df8e25a',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '563f3f4f-738d-4b6a-acd6-103fdc611e94',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': '2258cb3f-7db4-4269-b5c9-6e146868823d',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'f

adam, lr=0.1    -> test accuracy per epoch: [55.81, 58.7, 62.03, 58.56, 56.7]
adam, lr=0.01   -> test accuracy per epoch: [95.41, 96.71, 96.23, 96.74, 97.12]
adam, lr=0.001  -> test accuracy per epoch: [94.12, 96.1, 96.69, 97.21, 97.28]
adam, lr=0.0001 -> test accuracy per epoch: [89.17, 91.43, 92.67, 93.36, 93.9]


C:\Users\mateo\AppData\Roaming\Python\Python313\site-packages\jupyter_client\session.py:721: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


 sgd, lr=1      -> test accuracy per epoch: [9.8, 9.8, 9.8, 9.8, 9.8]
 sgd, lr=0.1    -> test accuracy per epoch: [96.33, 96.8, 97.62, 97.34, 97.69]
 sgd, lr=0.01   -> test accuracy per epoch: [92.44, 94.06, 95.41, 96.15, 96.44]
 sgd, lr=0.001  -> test accuracy per epoch: [78.96, 87.19, 89.37, 90.35, 91.05]


In [28]:
# Numeric summary: final test accuracy and mean gradient-to-weight ratio of each run.
print(f"{'optim':>5} {'lr':>7} {'final acc':>10} {'mean ratio':>11}")
for i, (opt_name, exp_lr) in enumerate(experiments):
    ratios = lr_visualizer.grad_to_weight_fig.data[i].y
    mean_ratio = float(np.mean(ratios)) if len(ratios) else float("nan")
    print(f"{opt_name:>5} {exp_lr:>7g} {results[(opt_name, exp_lr)][-1]:>10.2f} {mean_ratio:>11.2e}")

optim      lr  final acc  mean ratio
 adam     0.1      56.70    1.84e-04
 adam    0.01      97.12    6.81e-05
 adam   0.001      97.28    3.49e-05
 adam  0.0001      93.90    6.34e-06
  sgd       1       9.80         nan
  sgd     0.1      97.69    3.46e-03
  sgd    0.01      96.44    7.84e-04
  sgd   0.001      91.05    8.65e-05


### Conclusions

| Optimizer | lr | Final test acc. | What happens |
|---|---|---|---|
| Adam | 1e-1 | 56.7% | **Too large**: the loss oscillates and never converges |
| Adam | 1e-2 | 97.1% | Good, learns fast |
| Adam | 1e-3 | 97.3% | **Best for Adam**, smoothest curves (the usual Adam default) |
| Adam | 1e-4 | 93.9% | Too small: learns slowly, needs more epochs |
| SGD + momentum | 1 | 9.8% | **Diverges**: the loss becomes `NaN` and the net predicts a single class (~10% = chance) |
| SGD + momentum | 1e-1 | 97.7% | **Best for SGD** |
| SGD + momentum | 1e-2 | 96.4% | Good but slower |
| SGD + momentum | 1e-3 | 91.1% | Too slow |

**Key takeaways:**
1. There is a "sweet spot" for the learning rate: too large makes training unstable or makes it diverge; too small makes convergence very slow.
2. **Good learning rates differ between optimizers**: SGD with momentum works with learning rates ~10-100x larger than Adam (1e-1 vs. 1e-3).
3. **Gradient-to-weight ratio for SGD**: the actual update is `lr * grad` (plus momentum), so the ratio reflects the real step size. The best SGD run (lr=0.1) has a ratio of ~3e-3, close to the ~1e-3 rule of thumb; with lr=1e-3 it is ~1e-4 and learning is slow.
4. **Caveat for Adam**: the visualizer computes `lr * grad`, but Adam **normalizes** the gradient (each step is ≈ lr per parameter, regardless of the gradient's magnitude). That is why Adam with lr=0.1 shows a small ratio (~2e-4) even though training is unstable: for Adam this metric underestimates the real step size.